In [ ]:
import pandas as pd
from sklearn.metrics import f1_score, confusion_matrix, precision_score, recall_score
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def evaluate_dataframe(df: pd.DataFrame, title_prefix: str = ""):
    # bandaid fix: need to change the question list to use 'sleep disturbance' instead of sleep
    df["Var1"] = df.apply(lambda row: row["Var1"] if row["Var1"] != "Sleep" else "Sleep disturbance", axis=1)
    df["Var2"] = df.apply(lambda row: row["Var2"] if row["Var2"] != "Sleep" else "Sleep disturbance", axis=1)
    edge_list = pd.read_csv("../data/expert_edges_latest.csv")
    df = pd.merge(df, edge_list, how='outer', on=["Var1", "Var2"], indicator=True)
    df["Label"] = df["_merge"] == "both"

    metrics = ["Plausibility", "Association", "Temporality"]
    
    for metric in metrics:
        print(f"\n=== {metric} ===")
        y_true = df["Label"]
        y_pred = df[metric]
        
        # Calculate metrics
        f1 = f1_score(y_true, y_pred)
        tpr = recall_score(y_true, y_pred)  # TPR = Recall = TP/(TP+FN)
        precision = precision_score(y_true, y_pred, zero_division=0)
        fdr = 1 - precision if precision > 0 else 1  # FDR = FP/(TP+FP) = 1 - Precision
        
        # Confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        tn, fp, fn, tp = cm.ravel()
        
        print(f'F1-Score: {f1:.4f}')
        print(f'True Positive Rate (Recall): {tpr:.4f}')
        print(f'False Discovery Rate: {fdr:.4f}')
        print(f'Confusion Matrix:')
        print(f'  True Negatives:  {tn:4d}    False Positives: {fp:4d}')
        print(f'  False Negatives: {fn:4d}    True Positives:  {tp:4d}')
        
        # Create and save confusion matrix figure
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                    xticklabels=['Predicted Negative', 'Predicted Positive'],
                    yticklabels=['Actual Negative', 'Actual Positive'])
        
        title = f"{metric}"
        if title_prefix:
            title = f"{title_prefix} - {metric}"
        
        plt.title(f'{title} Confusion Matrix')
        plt.ylabel('Actual')
        plt.xlabel('Predicted')
        
        # Save figure
        filename = f"figs/{title.replace(' ', '_').replace('-', '_').lower()}_confusion_matrix.png"
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        plt.close()
        print(f'Confusion matrix saved as: {filename}')

In [ ]:
df = pd.read_csv("predictions/low_level_ontology.csv")

In [ ]:
evaluate_dataframe(df, "Low Level")

In [ ]:
df = pd.read_csv("predictions/high_level_ontology.csv")

In [ ]:
evaluate_dataframe(df, "High Level")